In [ ]:
import pandas as pd
from modelens import ClassificationAnalyzer, classification_models
from sklearn.datasets import fetch_openml

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [ ]:
raw_data = pd.read_csv("dataset.csv")

df = raw_data.copy()

In [ ]:
df = df.drop_duplicates()

In [ ]:
analyzer = ClassificationAnalyzer(df, target="target")

#### 1 - EDA + ETL

In [ ]:
# Check Invalid Data
analyzer.info()

In [ ]:
analyzer.reinit(df=df, target="class")

In [ ]:
df.head()

#### 2 - Comparing Models

In [ ]:
target = "target"
X = df.drop(columns=[target])
y = df[target]
features = df.columns.drop(target)

In [ ]:
analyzer.correlation()

In [ ]:
analyzer.vif(features=features)

In [ ]:
analyzer.permutation_importance()

In [ ]:
models = classification_models()

# analyzer.compare_models(
#     models=models,
#     features=features,
#     export_html=True,
#     file_name="01-base-model-comparing",
# )
# selected_model = CatBoost, SVM

In [ ]:
# from catboost import CatBoostClassifier

# catboost = CatBoostClassifier(random_state=42)
# analyzer.evaluate_feature_removal_combinations(
#     model=catboost,
#     features=features,
#     candidates=features,
#     export_html=True,
#     file_name="catboost",
# )

In [ ]:
# from sklearn.svm import SVC

# svm = SVC(random_state=42)

# analyzer.evaluate_feature_removal_combinations(
#     model=svm,
#     features=features,
#     candidates=features,
#     export_html=True,
#     file_name="svm",
# )

In [ ]:
# from sklearn.svm import SVC

# selected_model = SVC(random_state=42)

# param_grid = {
#     "max_depth": [3, 5, 7, 10, 15, None],
#     "min_samples_split": [2, 5, 10, 20],
#     "min_samples_leaf": [1, 2, 5, 10],
#     "criterion": ["gini", "entropy", "log_loss"],
#     "class_weight": [None, "balanced"],
# }
# # analyzer.tune_model(
# #     model=selected_model,
# #     features=features,
# #     param_grid=param_grid,
# #     export_html=True,
# #     file_name="decision_tree",
# # )

In [ ]:
# analyzer.evaluate_single_feature_removal(
#     model=selected_model,
#     features=features,
#     export_html=True,
#     file_name="decision_tree",
# )
# # Remove ...
# remove_features = [""]

In [ ]:
# analyzer.evaluate_feature_removal_combinations(
#     model=selected_model,
#     features=features,
#     candidates=features,
#     export_html=True,
#     file_name="decision_tree",
# )
# Remove

In [ ]:
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# optimized_features = features.drop([])
selected_model = RandomForestClassifier()

# analyzer.compare_feature_sets(
#     model=selected_model,
#     feature_sets={
#         "All Features": features,
#         "Optimized Features": optimized_features,
#     },
# )

In [ ]:
models = classification_models()
for name, model in models.items():
    print("-" * 10)
    print(name)
    print("-" * 10)
    analyzer.analyze_prediction_group(model=selected_model, features=features)

In [ ]:
# هر فیچر به تنهایی چقدر میتونه تارگت رو تفکیک کنه
import pandas as pd
from sklearn.metrics import roc_auc_score

results = []

for feature in features:
    auc = roc_auc_score(df[target], df[feature])

    # اگر رابطه برعکس باشد
    auc = max(auc, 1 - auc)

    results.append(
        {
            "Feature": feature,
            "ROC-AUC": auc,
        }
    )

feature_auc = pd.DataFrame(results).sort_values(
    "ROC-AUC",
    ascending=False,
)

display(feature_auc)

In [ ]:
feature_sets = {
    "Top 2": [
        "cp",
        "thalach",
    ],
    "Top 4": [
        "cp",
        "thalach",
        "ca",
        "oldpeak",
    ],
    "Top 6": [
        "cp",
        "thalach",
        "ca",
        "oldpeak",
        "thal",
        "exang",
    ],
    "Top 7": [
        "cp",
        "thalach",
        "ca",
        "oldpeak",
        "thal",
        "exang",
        "slope",
    ],
    "All Features": features,
}
analyzer.compare_feature_sets(
    model=selected_model,
    feature_sets=feature_sets,
    export_html=True,
    file_name="feature_sets_auc",
)

'✓ Report generated successfully: html_reports/compare_feature_sets/feature_sets_auc.html'

In [53]:
# FE
df_fe = df.copy()

df_fe["ca_thal"] = df_fe["ca"] * df_fe["thal"]
df_fe["ca_oldpeak"] = df_fe["ca"] * df_fe["oldpeak"]
df_fe["thal_oldpeak"] = df_fe["thal"] * df_fe["oldpeak"]
df_fe["exang_oldpeak"] = df_fe["exang"] * df_fe["oldpeak"]
df_fe["age_thalach"] = df_fe["age"] * df_fe["thalach"]

features_fe = list(features) + [
    "ca_thal",
    "ca_oldpeak",
    "thal_oldpeak",
    "exang_oldpeak",
    "age_thalach",
]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

selected_model = RandomForestClassifier()
analyzer.reinit(df=df_fe,target=target)
analyzer.compare_feature_sets(
    model=selected_model,
    feature_sets={
        "All Features": features,
        "Optimized Features": features_fe,
    },
)

'✓ Report generated successfully: html_reports/compare_feature_sets/feature_sets_comparison.html'

In [56]:
# one hot numerical encoded
df_encoded = pd.get_dummies(
    df,
    columns=[
        "cp",
        "restecg",
        "slope",
        "thal",
    ],
    drop_first=False,
    dtype=int,
)

features_encoded = df_encoded.columns.drop(target).tolist()

In [58]:
from sklearn.ensemble import RandomForestClassifier

selected_model = RandomForestClassifier(random_state=42)

# Original
analyzer.reinit(df=df, target=target)

analyzer.compare_feature_sets(
    model=selected_model,
    feature_sets={
        "Original": list(features),
    },
    file_name="original",
)

# One-Hot
analyzer.reinit(df=df_encoded, target=target)

analyzer.compare_feature_sets(
    model=selected_model,
    feature_sets={
        "One-Hot": features_encoded,
    },
    file_name="one_hot",
)

'✓ Report generated successfully: html_reports/compare_feature_sets/one_hot.html'

In [59]:
# FE2
df_fe = df.copy()

df_fe["thalach_age_ratio"] = (
    df_fe["thalach"] / (220 - df_fe["age"])
)

features_fe = list(features) + [
    "thalach_age_ratio",
]

In [61]:
from sklearn.ensemble import RandomForestClassifier

selected_model = RandomForestClassifier()
analyzer.reinit(df=df_fe,target=target)
analyzer.compare_feature_sets(
    model=selected_model,
    feature_sets={
        "All Features": features,
        "Optimized Features": features_fe,
    },
    file_name="age_ratio"
)

'✓ Report generated successfully: html_reports/compare_feature_sets/age_ratio.html'